In [0]:
# define var, lib
from pyspark.sql.functions import lower, regexp_replace
from pyspark.sql.functions import col, sum as _sum, when, trim

bronze_table = 'training.sch.bronze_games_ranking'
silver_table = 'training.sch.silver_games_ranking'

In [0]:
bronze_df = spark.table(bronze_table)

bronze_df.display()

game_name,genre,rank_type,rank
Counter-Strike 2,Action,Sales,1
"Warhammer 40,000: Space Marine 2",Action,Sales,2
Cyberpunk 2077,Action,Sales,3
Black Myth: Wukong,Action,Sales,4
ELDEN RING,Action,Sales,5
PUBG: BATTLEGROUNDS,Action,Sales,6
DRAGON BALL: Sparking! ZERO,Action,Sales,7
Apex Legends™,Action,Sales,8
Dota 2,Action,Sales,9
Party Animals,Action,Sales,10


In [0]:
%skip
# bronze_df 각 컬럼별 null/빈문자열 건수 확인
null_check = bronze_df.select(
    _sum(when(col("game_name").isNull(), 1).otherwise(0)).alias("game_name_null"),
    _sum(when(trim(col("game_name")) == "", 1).otherwise(0)).alias("game_name_blank"),
    _sum(when(col("genre").isNull(), 1).otherwise(0)).alias("genre_null"),
    _sum(when(col("rank_type").isNull(), 1).otherwise(0)).alias("rank_type_null"),
    _sum(when(col("rank").isNull(), 1).otherwise(0)).alias("rank_null"),
    _sum(when(col("rank").cast("int").isNull(), 1).otherwise(0)).alias("rank_cast_fail")
)

null_check.display()

game_name_null,game_name_blank,genre_null,rank_type_null,rank_null,rank_cast_fail
0,0,0,0,0,0


In [0]:
# bronze_df -> silver_df
# 'game_name' replace space with underscore

silver_df = bronze_df.withColumn("game_name", regexp_replace(lower("game_name"), " ", "_")) \
    .withColumn("rank", bronze_df["rank"].cast("int"))

silver_df.write.mode('overwrite').saveAsTable(silver_table)

game_name,genre,rank_type,rank
counter-strike_2,Action,Sales,1
"warhammer_40,000:_space_marine_2",Action,Sales,2
cyberpunk_2077,Action,Sales,3
black_myth:_wukong,Action,Sales,4
elden_ring,Action,Sales,5
pubg:_battlegrounds,Action,Sales,6
dragon_ball:_sparking!_zero,Action,Sales,7
apex_legends™,Action,Sales,8
dota_2,Action,Sales,9
party_animals,Action,Sales,10
